# LEAD-EEG on the 176-recording dementia-vs-healthy cohort (Paper 8, Paper 2 Study B)
CPU runtime. Runtime → Run all. Downloads the denoised derivatives of OpenNeuro ds004504 (eyes-closed) and ds006036 (photic stimulation), recorded from the same 88 participants.

In [ ]:
!git clone -q https://github.com/aftab802/lead-eeg
%cd lead-eeg
!pip -q install mne pymatreader imbalanced-learn eeglabio awscli

In [ ]:
for ds in ['ds004504', 'ds006036']:
    !aws s3 sync --no-sign-request --only-show-errors s3://openneuro.org/{ds} data/{ds} --exclude '*' --include 'participants.tsv' --include 'dataset_description.json' --include 'derivatives/*'
!ls data/ds004504/derivatives | wc -l; ls data/ds006036/derivatives | wc -l; ls data/ds006036/derivatives/sub-001/eeg

In [ ]:
!python scripts/run_lead_cohort176.py --ds004504 data/ds004504 --ds006036 data/ds006036 2>&1 | grep -v Warning | tail -8

In [ ]:
import glob, json
r = json.load(open(sorted(glob.glob('results/lead_cohort176_*.json'))[-1]))
out = {k: v for k, v in r.items() if k != 'runs'}
out['runs'] = {k: {'acc': v['accuracy_pooled'], 'fold_mean': v['accuracy_fold_mean'], 'fold_sd': v['accuracy_fold_sd'], 'cm': v['confusion_matrix'],
                   'dem': v['report']['1'], 'hc': v['report']['0']} for k, v in r['runs'].items()}
print('RESULTS_JSON_BEGIN'); print(json.dumps(out)); print('RESULTS_JSON_END')